# Paper 4 provider-recovery development starter

Select a **CUDA GPU** and run cells in order. This is a separately declared **Wikipedia + Wikidata, A-OKVQA TRAIN-development** experiment following archived ConceptNet HTTP 502 failures. It does not reproduce the original three-source protocol. There is no Drive mount, compute purchase, model training or held-out benchmark evaluation.

The adopted, pinned Qwen2.5-VL-3B generator and Qwen2.5-1.5B critic remain frozen. One real question is the first execution gate; a successful gate permits the bounded 50-question run. Internally valid execution does not establish a correct answer, novelty or PhD completion.

Commands retain complete stdout/stderr and show a heartbeat every 30 seconds at most. Download the final archive, including failures. Temporary Colab storage can disappear after disconnect. The protected full run may require **30 minutes or more**; availability and actual throughput determine its duration.

The previous larger development attempt was not validated: HTTP 429 availability failures and strict JSON contract failures remained. This starter preserves the strict parser; Markdown fences and prose are not silently converted into valid answers. Notebook compilation is software validation, not evidence that all 50 cases pass.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, importlib.metadata, json, os, platform, subprocess, sys, time, uuid, zipfile
import fcntl
SOURCE_COMMIT = 'eefa4aa07e41033b77d13779e834e1e0e6f841db'
REPOSITORY = 'https://github.com/junnubabu-ctrl/paper4-selective-kbvqa.git'
RUN_ROOT = Path('/content/JB_Paper4/provider_recovery_starter_eefa4aa')
REPO = Path('/content/paper4_provider_recovery_starter_eefa4aa')
PILOT = RUN_ROOT / 'prepared_train50'
ONE_OUT = RUN_ROOT / 'two_source_one_image_v1'
FIFTY_OUT = RUN_ROOT / 'two_source_pilot50_v1'
SOURCE_LEDGER = RUN_ROOT / 'source_policy.json'
RUN_ROOT.mkdir(parents=True, exist_ok=True)
(RUN_ROOT / 'commands').mkdir(exist_ok=True)
_RUNNING = globals().get('_RUNNING', {})

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.part')
    with tmp.open('w', encoding='utf-8') as stream:
        json.dump(value, stream, indent=2, ensure_ascii=False, allow_nan=False)
        stream.write('\n')
        stream.flush()
        os.fsync(stream.fileno())
    os.replace(tmp, path)

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def process_identity(pid):
    try:
        fields = Path(f'/proc/{int(pid)}/stat').read_text().rsplit(')', 1)[1].split()
        return {'state': fields[0], 'start_ticks': fields[19]}
    except (OSError, ValueError, IndexError):
        return None

def process_is_live(record):
    if record.get('status') != 'RUNNING' or not isinstance(record.get('pid'), int):
        return False
    found = process_identity(record['pid'])
    if found is not None:
        if found['state'] == 'Z':
            return False
        expected = record.get('pid_start_ticks')
        return expected is None or found['start_ticks'] == expected
    # Conservatively block duplicates when /proc is unavailable but the PID exists.
    # A missing start identity is not represented as a confirmed PID identity.
    try:
        os.kill(record['pid'], 0)
        return True
    except ProcessLookupError:
        return False
    except PermissionError:
        return True

def active_jobs():
    records = [json.loads(p.read_text()) for p in (RUN_ROOT / 'commands').glob('*/process.json')]
    return [record for record in records if process_is_live(record)]

def counts(out_dir):
    return ({name: len(list((Path(out_dir) / name).glob('*.json')))
             for name in ['contexts', 'candidates', 'critics']} if out_dir else {})

def show_new_log(path, offsets, label):
    with Path(path).open('rb') as stream:
        stream.seek(offsets.get(label, 0))
        new = stream.read()
        offsets[label] = stream.tell()
    if new:
        print('[' + label + ']\n' + new[-6000:].decode('utf-8', errors='replace'), flush=True)
        if len(new) > 6000:
            print('Earlier output retained in ' + str(path), flush=True)

def run_live(label, args, *, cwd=None, check=True, out_dir=None):
    """Synchronous monitoring with full logs and guarded duplicate launches."""
    assert label.replace('_', '').isalnum()
    argv = [str(x) for x in args]
    with (RUN_ROOT / 'commands' / 'execution.lock').open('a') as lock:
        try:
            fcntl.flock(lock.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError:
            raise RuntimeError('An execution is already being monitored; do not launch another child')
        for existing in active_jobs():
            raise RuntimeError(f'{existing["label"]} is active (PID {existing["pid"]}). '
                               f'Use monitor_existing({existing["label"]!r}); do not start another command.')
        attempt = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ') + '_' + uuid.uuid4().hex[:8]
        folder = RUN_ROOT / 'commands' / (label + '_' + attempt)
        folder.mkdir()
        stdout_path, stderr_path = folder / 'stdout.log', folder / 'stderr.log'
        record = {'schema': 'paper4-notebook-command-v1', 'label': label, 'argv': argv,
                  'cwd': str(Path(cwd).resolve()) if cwd is not None else str(Path.cwd()),
                  'requested_source_commit': SOURCE_COMMIT, 'started_utc': utc_now(),
                  'status': 'LAUNCHING', 'out_dir': str(out_dir) if out_dir else None,
                  'stdout_path': str(stdout_path), 'stderr_path': str(stderr_path),
                  'returncode': None, 'finished_utc': None}
        meta_path = folder / 'process.json'
        save_json(meta_path, record)
        started, offsets = time.monotonic(), {}
        with stdout_path.open('wb') as stdout, stderr_path.open('wb') as stderr:
            try:
                process = subprocess.Popen(argv, cwd=cwd, stdout=stdout, stderr=stderr,
                                           env={**os.environ, 'PYTHONUNBUFFERED': '1'})
            except BaseException as error:
                record.update(status='LAUNCH_FAILED', finished_utc=utc_now(), error=f'{type(error).__name__}: {error}')
                save_json(meta_path, record)
                raise
            found = process_identity(process.pid)
            record.update(status='RUNNING', pid=process.pid,
                          pid_start_ticks=found['start_ticks'] if found else None)
            save_json(meta_path, record)
            _RUNNING[label] = (process, meta_path)
            print(json.dumps({'launched': label, 'pid': process.pid, 'argv': argv,
                              'source_commit': SOURCE_COMMIT, 'logs': str(folder)}), flush=True)
            try:
                while True:
                    try:
                        process.wait(timeout=30)
                    except subprocess.TimeoutExpired:
                        pass
                    show_new_log(stdout_path, offsets, 'stdout')
                    show_new_log(stderr_path, offsets, 'stderr')
                    print(json.dumps({'heartbeat_utc': utc_now(), 'label': label,
                          'elapsed_s': round(time.monotonic() - started, 1),
                          'returncode': process.poll(), 'archived_records': counts(out_dir)}), flush=True)
                    if process.poll() is not None:
                        break
                record.update(status='FINISHED', returncode=process.returncode,
                              finished_utc=utc_now(), elapsed_s=time.monotonic() - started)
                save_json(meta_path, record)
                _RUNNING.pop(label, None)
            except BaseException as error:
                record.update(monitor_interrupted_utc=utc_now(), monitor_error=f'{type(error).__name__}: {error}')
                save_json(meta_path, record)
                print(f'Monitor interrupted; child is not silently killed. Use monitor_existing({label!r}).', flush=True)
                raise
        if check and record['returncode'] != 0:
            raise subprocess.CalledProcessError(record['returncode'], argv)
        return record

def monitor_existing(label):
    """Resume monitoring an existing child without launching anything."""
    records = [(p, json.loads(p.read_text())) for p in (RUN_ROOT / 'commands').glob('*/process.json')]
    matches = [(p, r) for p, r in records if r.get('label') == label and r.get('status') == 'RUNNING']
    assert len(matches) == 1, f'Expected one unfinished {label} record, found {len(matches)}'
    meta_path, record = matches[0]
    offsets = {}
    while process_is_live(record):
        process = _RUNNING.get(label, (None, None))[0]
        if process is not None:
            try:
                process.wait(timeout=30)
            except subprocess.TimeoutExpired:
                pass
        else:
            time.sleep(30)
        show_new_log(record['stdout_path'], offsets, 'stdout')
        show_new_log(record['stderr_path'], offsets, 'stderr')
        print(json.dumps({'heartbeat_utc': utc_now(), 'label': label,
                          'archived_records': counts(record.get('out_dir'))}), flush=True)
    process = _RUNNING.get(label, (None, None))[0]
    record.update(status='FINISHED' if process is not None else 'EXIT_STATUS_UNAVAILABLE',
                  returncode=process.poll() if process is not None else None, finished_utc=utc_now())
    save_json(meta_path, record)
    _RUNNING.pop(label, None)
    print('Process ended. Rerun its gate to read immutable records, then audit and archive.')
    return record

assert not active_jobs(), 'A previous child is active. Resume monitor_existing(label) before rerunning setup.'
import torch
assert torch.cuda.is_available(), 'Select a CUDA GPU runtime. No predictions generated.'
hardware = {'checked_utc': utc_now(), 'python': sys.version, 'platform': platform.platform(),
            'torch': torch.__version__, 'cuda': torch.version.cuda,
            'devices': [{'index': i, 'name': torch.cuda.get_device_name(i),
                         'total_memory_bytes': int(torch.cuda.get_device_properties(i).total_memory)}
                        for i in range(torch.cuda.device_count())], 'benchmark_result': False}
save_json(RUN_ROOT / 'hardware.json', hardware)
print(json.dumps(hardware, indent=2))
run_live('hardware', ['nvidia-smi'])


## 1. Verify exact code and dependencies

The checkout must be detached at the specified code commit with clean tracked files. Existing mismatched code is rejected rather than replaced. Model revisions come from the repository lock. Installation output and actual package versions are retained.

The source policy declares Wikipedia + Wikidata only. Wikimedia rate protection paces requests, retains HTTP errors and respects Retry-After. Individual waiting must remain bounded to 30 seconds; a newly observed HTTP 429 is not immediately retried. A future uncached request is permitted only after a short Retry-After (at most 30 seconds) is respected. A longer cooldown latches an actionable availability failure for the rest of that invocation; it does not permit further requests or a bypass. The 50-case continuation flag preserves diagnostic failures; it does not claim provider availability or 100% success.


In [ ]:
assert not active_jobs(), 'A child is active. Resume monitoring before checkout or installation.'
if not REPO.exists():
    run_live('clone', ['git', 'clone', REPOSITORY, REPO])
    run_live('checkout', ['git', '-C', REPO, 'checkout', '--detach', SOURCE_COMMIT])
actual = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
assert actual == SOURCE_COMMIT, 'Existing checkout differs. Use a fresh declared folder/runtime.'
assert subprocess.run(['git', '-C', str(REPO), 'symbolic-ref', '-q', 'HEAD'], capture_output=True).returncode != 0
assert not subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain',
                                   '--untracked-files=no'], text=True).strip(), 'Tracked source changed.'
run_live('install', [sys.executable, '-m', 'pip', 'install', '-e', str(REPO) + '[vlm,dev]', 'transformers==4.57.1'])
packages = {name: importlib.metadata.version(name) for name in
            ['torch', 'transformers', 'accelerate', 'bitsandbytes', 'numpy', 'scikit-learn', 'pillow']}
(RUN_ROOT / 'source_commit.txt').write_text(actual + '\n', encoding='utf-8')
model_lock = json.loads((REPO / 'configs/models_20260915.json').read_text())
save_json(RUN_ROOT / 'verified_checkout.json', {'actual_commit': actual, 'detached': True,
    'tracked_source_clean': True, 'packages': packages,
    'model_lock_sha256': sha256_file(REPO / 'configs/models_20260915.json'),
    'models': {key: model_lock[key] for key in ['generator', 'qwen']}})
print(json.dumps({'source_commit': actual, 'packages': packages,
                  'models': {key: model_lock[key] for key in ['generator', 'qwen']}}, indent=2))
policy = {'schema': 'paper4-development-source-policy-v1', 'stage': 'train-development',
    'source_set': ['wikipedia', 'wikidata'], 'omitted_sources': ['conceptnet'],
    'reason': 'Explicit two-source development recovery following an archived ConceptNet HTTP 502 failure.',
    'original_three_source_run_reproduced': False, 'benchmark_result': False,
    'protocol': 'explicit-two-source-development-v1',
    'knowledge_capture': 'live responses captured once per question; upstream releases not immutable',
    'references_role': 'offline evaluator only; never supplied as generator or retrieval input'}
if SOURCE_LEDGER.exists():
    assert json.loads(SOURCE_LEDGER.read_text()) == policy, 'Source policy changed; use a new folder.'
else:
    save_json(SOURCE_LEDGER, policy)
# Editable installation does not retroactively reload a running kernel's import paths.
sys.path.insert(0, str(REPO / 'src'))
import paper4_kbvqa
assert Path(paper4_kbvqa.__file__).resolve().is_relative_to((REPO / 'src').resolve()), \
    'A different checkout is already imported. Restart the runtime before executing this starter.'
from paper4_kbvqa.execution.fixed_candidate import read_verified, digest, file_digest

def assert_pinned_checkout():
    assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
    assert not subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain',
                                       '--untracked-files=no'], text=True).strip()


## 2. Prepare and verify 50 development image–question pairs

Only the official A-OKVQA TRAIN member and the selected 50 COCO train2017 JPEGs are prepared. Original dimensions and hashes are retained. **`coco-s3-path`** addresses the official COCO image bucket over HTTPS with normal certificate checks; TLS verification is never disabled.

The label-free inference manifest is separate from offline reference answers. Identity validation can read reference IDs, but reference answers are never generator prompts or retrieval input. Reserve all selected image groups from subsequent calibration and threshold sets. A failed download/preflight blocks inference.


In [ ]:
assert not active_jobs(), 'A child is active. Resume monitoring before preparing data.'
DATA_READY = False
prepared = run_live('prepare_data', [sys.executable, REPO / 'scripts/prepare_restart_pilot.py',
    '--out-dir', PILOT, '--download-annotations', '--download-images',
    '--image-transport', 'coco-s3-path', '--timeout', '30'], cwd=REPO, check=False)
if prepared['returncode'] == 0:
    checked = run_live('preflight', [sys.executable, REPO / 'scripts/restart_preflight.py',
        '--pilot-dir', PILOT, '--out', RUN_ROOT / 'preflight.json'], cwd=REPO, check=False)
    preflight_path = RUN_ROOT / 'preflight.json'
    preflight_diagnostic = {'checked_utc': utc_now(), 'returncode': checked['returncode'],
                            'report_exists': preflight_path.exists(), 'errors': []}
    try:
        if not preflight_path.exists():
            raise ValueError('Preflight process produced no report; see retained stdout/stderr.')
        preflight = json.loads(preflight_path.read_text())
        if not isinstance(preflight, dict):
            raise ValueError('Preflight report must be a JSON object.')
        preflight_diagnostic['report_status'] = preflight.get('status')
        DATA_READY = checked['returncode'] == 0 and preflight.get('status') == 'READY_FOR_ONE_IMAGE_GATE'
    except (OSError, ValueError, TypeError) as error:
        preflight_diagnostic['errors'].append(f'{type(error).__name__}: {error}')
        DATA_READY = False
    save_json(RUN_ROOT / 'preflight_gate_diagnostic.json', preflight_diagnostic)
if DATA_READY:
    from paper4_kbvqa.data.restart_pilot import validate_prepared_pilot
    verified_data = validate_prepared_pilot(PILOT, require_images=True)
    print(json.dumps({'status': 'READY_FOR_ONE_IMAGE_GATE', 'questions': len(verified_data['rows']),
        'distinct_images': len({row['metadata']['image_id'] for row in verified_data['rows']}),
        'partition': 'TRAIN development only', 'benchmark_result': False}, indent=2))
else:
    print('BLOCKED: data/preflight failed. Skip inference; audit and archive these failures below.')


## 3. Audit helper and one-image gate

The first real case uses the pinned generator, NF4 configuration, critic and declared two-source policy. The CLI stops at an invalid record. Raw outputs, strict JSON parsing, grounding, evidence, citations and resource sessions remain archived.

The checksum audit verifies image/input/run/data relationships and exact B2/B3 candidate reuse. A failed case is not repaired using reference answers. Changed code, prompts, resolution, models or sources require a new declared output directory.


In [ ]:
from paper4_kbvqa.execution.fixed_candidate import read_verified, digest, file_digest, inference_identity
from paper4_kbvqa.data.manifest import VQASample
from paper4_kbvqa.data.restart_pilot import validate_prepared_pilot

def audit_run(output):
    output = Path(output)
    audit = {'output': str(output), 'stage': 'train-development', 'benchmark_result': False,
             'scientific_claims_validated': False, 'audit_pass': False, 'errors': [], 'records': []}
    if not output.exists():
        audit['status'] = 'NOT_STARTED'
        return audit
    try:
        data = validate_prepared_pilot(PILOT, require_images=True)
        run = read_verified(output / 'run_identity.json')
        run_sha = digest(run)
        assert run['stage'] == 'train-development' and run['benchmark_result'] is False
        expected_data = {'mode': 'validated_prepared_train_development',
            'source_manifest_sha256': data['source_manifest_sha256'],
            'image_inventory_sha256': data['image_inventory_sha256'],
            'inference_manifest_sha256': data['source']['inference_manifest_sha256'],
            'source_split': 'train', 'partition_role': 'development_only'}
        assert run['run_config']['data_identity'] == expected_data, 'Data identity mismatch'
        assert run['source_identity']['sources'] == ['wikipedia', 'wikidata']
        assert run['source_identity']['source_ledger_sha256'] == file_digest(SOURCE_LEDGER)
        assert run['run_config']['model_lock_sha256'] == file_digest(REPO / 'configs/models_20260915.json')
        source_paths = sorted((REPO / 'src').rglob('*.py')) + sorted((REPO / 'scripts').glob('*.py'))
        hashes = {str(p.relative_to(REPO)): file_digest(p) for p in source_paths}
        assert run['run_config']['code_identity']['modified_source_sha256'] == digest(hashes)
        assert run['run_config']['plan']['models'] == {key: model_lock[key] for key in ['generator', 'qwen']}
        declared_count = run['run_config']['plan']['max_samples']
        assert isinstance(declared_count, int) and 1 <= declared_count <= 50
        assert len(run['inputs']) == declared_count, 'Declared input count mismatch'
        expected_inputs = [inference_identity(VQASample(str(row['question_id']),
            row['image_path'], row['question'], metadata=row['metadata']))
            for row in data['rows'][:declared_count]]
        assert run['inputs'] == expected_inputs, 'Prepared input content or prefix order mismatch'
        inputs = {str(item['question_id']): item for item in run['inputs']}
        assert len(inputs) == len(run['inputs'])
        assert run['input_order_sha256'] == digest(run['inputs'])
        expected_stems = {hashlib.sha256(qid.encode()).hexdigest() for qid in inputs}
        paths = {name: {p.stem: p for p in (output / name).glob('*.json')}
                 for name in ['contexts', 'candidates', 'critics']}
        for name, collection in paths.items():
            assert set(collection) <= expected_stems, f'Foreign {name} records'
        assert set(paths['contexts']) == set(paths['candidates']) == set(paths['critics']), 'Incomplete triplets'
        matched = []
        for identity in run['inputs']:
            qid = str(identity['question_id'])
            stem = hashlib.sha256(qid.encode()).hexdigest()
            if stem not in paths['contexts']:
                continue
            context, candidate, critic = [read_verified(paths[name][stem])
                                          for name in ['contexts', 'candidates', 'critics']]
            for record in [context, candidate, critic]:
                assert str(record['question_id']) == qid and record['run_identity_sha256'] == run_sha
                assert record['stage'] == 'train-development'
            assert context['input_identity'] == candidate['input_identity'] == identity
            assert file_digest(identity['image_path']) == identity['image_sha256']
            assert context['source_identity'] == run['source_identity']
            assert candidate['context_record_sha256'] == digest(context)
            assert critic['candidate_record_sha256'] == digest(candidate)
            assert critic['candidate_sha256'] == candidate.get('candidate_sha256')
            assert critic['context_sha256'] == candidate.get('context_sha256')
            if 'context' in context:
                assert context['context_sha256'] == digest(context['context'])
            if candidate.get('candidate_eligible'):
                assert candidate['context'] == context['context']
                assert candidate['context_sha256'] == context['context_sha256']
                assert candidate['candidate_sha256'] == digest(candidate['candidate'])
                assert candidate['candidate']['generator_result_sha256'] == digest(candidate['generator_result'])
                assert candidate['source_identity'] == run['source_identity']
            if critic['status'] == 'VALID_MATCHED_DEVELOPMENT_RECORD':
                for variant in ['B2', 'B3']:
                    result = critic[variant]
                    assert result['answer'] == candidate['candidate']['answer']
                    assert result['candidate_sha256'] == candidate['candidate_sha256']
                    assert result['context_sha256'] == candidate['context_sha256']
                    assert result['calibrated_confidence'] is result['threshold'] is result['abstain'] is None
                assert critic['benchmark_result'] is False and critic['calibration_implemented'] is False
            generated = candidate.get('generator_result', {})
            selected_ids = {item['evidence_id'] for item in context.get('context', {}).get('ordered_evidence', [])}
            raw_ids = generated.get('raw_evidence_ids', generated.get('supporting_evidence_ids', []))
            raw_ids = [x for x in raw_ids if isinstance(x, str)] if isinstance(raw_ids, list) else []
            unknown = [eid for eid in raw_ids if eid not in selected_ids]
            repeated = [eid for eid in dict.fromkeys(raw_ids) if raw_ids.count(eid) > 1]
            audit['records'].append({'question_id': qid, 'question': identity['question'],
                'candidate_status': candidate['status'], 'critic_status': critic['status'],
                'answer': generated.get('answer'), 'raw_text': generated.get('raw_text'),
                'parser_status': generated.get('parser_status'),
                'grounding_parser_status': context.get('visual_grounding_record', {}).get('parser_status'),
                'selected_evidence_count': len(selected_ids), 'unknown_citation_occurrences': unknown,
                'repeated_citation_values': repeated, 'critic_result': critic.get('critic_result'),
                'errors': {'context': context['errors'], 'candidate': candidate['errors'], 'critic': critic['errors']}})
            matched.append({'question_id': qid, 'context_record_sha256': digest(context),
                'candidate_record_sha256': digest(candidate), 'critic_record_sha256': digest(critic),
                'candidate_status': candidate['status'], 'critic_status': critic['status'],
                'B2': critic.get('B2'), 'B3': critic.get('B3')})
        valid = sum(row['critic_status'] == 'VALID_MATCHED_DEVELOPMENT_RECORD' for row in audit['records'])
        attempted = len(audit['records'])
        audit.update(requested=len(inputs), attempted=attempted, valid=valid,
            invalid=attempted-valid, unattempted=len(inputs)-attempted,
            unknown_citation_occurrences=sum(len(r['unknown_citation_occurrences']) for r in audit['records']),
            questions_with_unknown_citations=sum(bool(r['unknown_citation_occurrences']) for r in audit['records']),
            data_identity=expected_data, run_identity_sha256=run_sha)
        if (output / 'pilot_summary.json').exists():
            summary = read_verified(output / 'pilot_summary.json')
            assert summary['run_identity_sha256'] == run_sha and summary['matched'] == matched
            for field in ['requested', 'attempted', 'valid', 'invalid', 'unattempted']:
                assert summary[field] == audit[field], f'Summary {field} mismatch'
            assert summary['benchmark_result'] is False and summary['scientific_claims_validated'] is False
            audit['status'] = summary['status']
        else:
            audit['status'] = 'INCOMPLETE_NO_SUMMARY'
        audit['audit_pass'] = True
    except Exception as error:
        audit['status'] = 'AUDIT_FAILED'
        audit['errors'].append(f'{type(error).__name__}: {error}')
    return audit

ONE_GATE_PASS = False
if DATA_READY:
    assert_pinned_checkout()
    if not (ONE_OUT / 'pilot_summary.json').exists():
        run_live('one_image', [sys.executable, REPO / 'scripts/run_fixed_candidate_pilot.py',
            '--pilot-dir', PILOT, '--out-dir', ONE_OUT, '--max-samples', '1',
            '--sources', 'wikipedia,wikidata', '--source-ledger', SOURCE_LEDGER],
            cwd=REPO, check=False, out_dir=ONE_OUT)
    else:
        print('Existing immutable summary; no duplicate inference launched.')
    one_audit = audit_run(ONE_OUT)
    ONE_GATE_PASS = (one_audit['audit_pass'] and one_audit.get('requested') == 1
        and one_audit.get('attempted') == one_audit.get('valid') == 1
        and one_audit.get('invalid') == one_audit.get('unattempted') == 0
        and one_audit['status'] == 'COMPLETED_VALID_DEVELOPMENT_PILOT')
    save_json(RUN_ROOT / 'one_image_gate_audit.json', one_audit)
    print(json.dumps({k: v for k, v in one_audit.items() if k != 'records'}, indent=2))
print('One-image gate:', 'PASS' if ONE_GATE_PASS else 'BLOCKED')
if not ONE_GATE_PASS:
    print('The 50-question run is skipped. Continue below to retain the complete failure archive.')


## 4. Bounded 50-question development execution

The same prepared data, source ledger and core models are reused. **`--continue-on-invalid`** preserves and attempts diagnostic failures within the bounded development protocol. Invalid outputs remain failures; a nonzero exit is retained. Rate protection does not permit repeated HTTP 429 requests or waits exceeding the declared cap.

The process below is synchronous, with visible 30-second heartbeats. Run All reaches the audit/download only after it ends. The runner does not train an own LLM, fit calibration, select abstention thresholds or evaluate held-out accuracy.


In [ ]:
if ONE_GATE_PASS:
    assert_pinned_checkout()
    gate = audit_run(ONE_OUT)
    assert gate['audit_pass'] and gate['requested'] == gate['attempted'] == gate['valid'] == 1
    assert gate['invalid'] == gate['unattempted'] == 0
    if not (FIFTY_OUT / 'pilot_summary.json').exists():
        run_live('development50', [sys.executable, REPO / 'scripts/run_fixed_candidate_pilot.py',
            '--pilot-dir', PILOT, '--out-dir', FIFTY_OUT, '--max-samples', '50',
            '--sources', 'wikipedia,wikidata', '--source-ledger', SOURCE_LEDGER,
            '--continue-on-invalid'], cwd=REPO, check=False, out_dir=FIFTY_OUT)
    else:
        print('Existing immutable 50-question summary; no duplicate inference launched.')
    if (FIFTY_OUT / 'pilot_summary.json').exists():
        fifty = read_verified(FIFTY_OUT / 'pilot_summary.json')
        print(json.dumps({k: v for k, v in fifty.items() if k != 'matched'}, indent=2))
    else:
        print('No completed summary. Inspect retained logs/resource sessions; do not claim all 50 completed.')
else:
    print('SKIPPED: one-image gate blocked. Failure records remain archived.')


## 5. Actual counts and diagnostics

Every context, candidate and critic is checksum verified. Actual requested/attempted/valid/invalid/unattempted counts are derived from archived records and reconciled with the summary. Unknown/repeated citation IDs remain visible rather than silently excluded.

B2 is whole-response token likelihood, including JSON/citation tokens; B3 is heuristic evidence-reliability fusion. Neither is calibrated correctness probability. The text critic reads extracted entities and evidence rather than the original image. **No development accuracy or benchmark metric is computed.**


In [ ]:
assert not active_jobs(), 'A child is active; resume monitor_existing(label) before auditing.'
execution_audit = {'schema': 'paper4-notebook-development-audit-v1', 'audited_utc': utc_now(),
    'source_commit': SOURCE_COMMIT, 'benchmark_result': False, 'calibration_implemented': False,
    'scientific_claims_validated': False, 'runs': [audit_run(p) for p in [ONE_OUT, FIFTY_OUT]]}
started = [r for r in execution_audit['runs'] if 'data_identity' in r]
execution_audit['cross_run_data_identity_equal'] = (
    len(started) == 2 and started[0]['data_identity'] == started[1]['data_identity'])
save_json(RUN_ROOT / 'development_execution_audit.json', execution_audit)
for audit in execution_audit['runs']:
    print(json.dumps({k: v for k, v in audit.items() if k != 'records'}, indent=2))
    for row in audit['records']:
        print(json.dumps({k: row[k] for k in ['question_id', 'candidate_status', 'critic_status',
            'answer', 'parser_status', 'selected_evidence_count', 'unknown_citation_occurrences', 'errors']},
            ensure_ascii=False))
print('These are execution counts, not correct/wrong answer counts or benchmark performance.')


## 6. Download complete evidence

Archive only after child processes end. The ZIP retains command metadata, full logs, identities, raw valid/invalid outputs, images, separate offline references, resource sessions and the audit. SHA256 detects changes; it is not an upstream signature.

If monitoring was interrupted, use **`monitor_existing('one_image')`** or **`monitor_existing('development50')`**; rerun its gate to read completed immutable records, then audit and download. Do not launch a duplicate process. An unknown child exit status after a kernel restart remains explicitly unknown.

Remaining scientific work includes protocol freezing, image-disjoint calibration/threshold subsets, policy fitting and locking, genuine held-out evaluation, paired uncertainty, controls/ablations and blinded evidence annotation where required. No 100% success, novelty or PhD-completion claim follows from this starter.


In [ ]:
assert not active_jobs(), 'Archive blocked while a child is active. Resume monitoring first.'
assert_pinned_checkout()
inventory = [{'path': str(p.relative_to(RUN_ROOT)), 'bytes': p.stat().st_size, 'sha256': sha256_file(p)}
    for p in sorted(RUN_ROOT.rglob('*')) if p.is_file() and not p.name.endswith('.part')
    and p.name != 'retained_artifact_inventory.json']
save_json(RUN_ROOT / 'retained_artifact_inventory.json', inventory)
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
archive = Path('/content') / ('paper4_provider_recovery_development_' + stamp + '.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(RUN_ROOT.rglob('*')):
        if path.is_file():
            bundle.write(path, str(path.relative_to(RUN_ROOT)))
print(json.dumps({'archive': str(archive), 'bytes': archive.stat().st_size,
    'sha256': sha256_file(archive), 'source_commit': SOURCE_COMMIT, 'benchmark_result': False}, indent=2))
from google.colab import files
files.download(str(archive))
